# ZemaVoice Stage 10 — Kids Voice Library
**Personal noncommercial prototype.** Six selectable youthful *cartoon-style effects* for Amharic speech. They are **not six genuine child speakers**: all use one adult pretrained Amharic voice (`facebook/mms-tts-amh`), altered by pitch and speed. This can sound synthetic. The base model has a **CC BY-NC 4.0** license; do not deploy commercially. Keep clips short for your first test.

Run the cells in order. Choose a preset, edit Amharic text, run Generate, and download the WAV.


In [ ]:
!pip -q install 'transformers>=4.44,<5' 'accelerate>=0.26' 'soundfile>=0.12' 'uroman>=1.3' 'librosa>=0.10' 'ipywidgets>=8'


In [ ]:
import torch, numpy as np, librosa, soundfile as sf
from pathlib import Path
from IPython.display import Audio, display
from transformers import VitsModel, VitsTokenizer, set_seed
from uroman import Uroman
import ipywidgets as widgets
from google.colab import files
model_id='facebook/mms-tts-amh'
tokenizer=VitsTokenizer.from_pretrained(model_id)
model=VitsModel.from_pretrained(model_id).eval()
device='cuda' if torch.cuda.is_available() else 'cpu'
model=model.to(device)
romanizer=Uroman()
sr=int(model.config.sampling_rate)
print('Kids Voice Library ready on',device,'|',sr,'Hz')


In [ ]:
presets={
 'Kiki — cheerful girl (3-style)': {'pitch':2.0,'rate':1.04},
 'Beni — playful boy (4-style)': {'pitch':1.0,'rate':1.06},
 'Mimi — gentle girl (2-style)': {'pitch':2.7,'rate':0.96},
 'Titi — curious boy (3-style)': {'pitch':1.6,'rate':1.02},
 'Lili — expressive girl (4-style)': {'pitch':2.3,'rate':1.09},
 'Nati — friendly boy (4-style)': {'pitch':0.8,'rate':0.98},
}
voice=widgets.Dropdown(options=list(presets),description='Character:',layout=widgets.Layout(width='65%'))
text_input=widgets.Textarea(value='ሰላም ጓደኞቼ! እንዴት ናችሁ?',description='Amharic:',layout=widgets.Layout(width='95%',height='110px'))
pitch_adjust=widgets.FloatSlider(value=0,min=-1.5,max=1.5,step=0.25,description='Pitch tweak:',continuous_update=False)
speed_adjust=widgets.FloatSlider(value=1.0,min=0.9,max=1.1,step=0.01,description='Speed tweak:',continuous_update=False)
display(widgets.VBox([voice,text_input,pitch_adjust,speed_adjust]))
print('Choose a character and edit text, then run the next cell. Age labels describe intended cartoon style, not real child voices.')


In [ ]:
amharic_text=text_input.value.strip()
assert 0<len(amharic_text)<=600,'Enter 1–600 characters for this test.'
romanized=romanizer.romanize_string(amharic_text)
assert romanized,'Romanization failed.'
inputs=tokenizer(romanized,return_tensors='pt').to(device)
set_seed(42)
with torch.inference_mode():
    base=model(**inputs).waveform[0].detach().cpu().float().numpy().astype(np.float32)
preset=presets[voice.value]
steps=preset['pitch']+pitch_adjust.value
rate=preset['rate']*speed_adjust.value
wave=librosa.effects.pitch_shift(base,sr=sr,n_steps=steps) if steps!=0 else base.copy()
wave=librosa.effects.time_stretch(wave,rate=rate) if abs(rate-1)>0.001 else wave
wave=np.clip(np.nan_to_num(wave),-0.98,0.98).astype(np.float32)
slug=voice.value.split(' — ')[0].lower()
out=Path(f'/content/zemavoice_stage10_{slug}.wav')
sf.write(out,wave,sr)
print('Voice:',voice.value,'| Duration:',round(len(wave)/sr,2),'seconds | Saved:',out)
display(Audio(wave,rate=sr))


In [ ]:
files.download(str(out))


## Notes
- This notebook generates **one voice at a time**. To create multiple character lines, select another preset and rerun the generation and download cells.
- All presets use the **same source speaker** and may not sound convincingly like distinct children. A truly natural multi-speaker library will require separately licensed, suitable voice models or recordings.
- You can import the resulting WAV files into Stage 7 for timed video audio synchronization. Lip-sync remains a separate unfinished feature.
